# 04. 지역 유형 군집화

K-Means로 시군구를 문화결핍 유형별로 분류하고
맞춤형 정책 추천을 도출합니다.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
from preprocessing import load_raw_data
from feature_engineering import build_region_features_from_raw
from scoring import add_scores
from clustering import add_clusters
from recommendation import add_recommendations
raw = load_raw_data()
features = build_region_features_from_raw(raw)
scored = add_scores(features)
clustered = add_clusters(scored)
result = add_recommendations(clustered)
print('군집별 시군구 수:')
print(result['cluster'].value_counts().sort_index())

## 4-1. 군집별 평균 지표

In [ ]:
cluster_profile = result.groupby('cluster')[
    ['supply_score','demand_score','access_penalty','mismatch_penalty','culture_gap_score']
].mean().round(1)
cluster_profile

## 4-2. 지역 유형 분포

In [ ]:
type_cnt = result['region_type'].value_counts()
type_cnt.plot.barh(figsize=(8, 4), color='#5b8fde')
plt.gca().invert_yaxis()
plt.xlabel('시군구 수')
plt.title('전국 문화결핍 지역 유형 분포')
plt.tight_layout()
plt.show()

## 4-3. 유형별 대표 지역

In [ ]:
for rtype in result['region_type'].unique():
    sub = result[result['region_type']==rtype].sort_values('culture_gap_score', ascending=False)
    rep = sub.head(3)['region_name'].tolist()
    print(f'[{rtype}] 대표 지역: {rep}')

## 4-4. 정책 추천 샘플

In [ ]:
top = result.sort_values('culture_gap_score', ascending=False).head(5)
for _, row in top.iterrows():
    print(f"\n■ {row['region_name']} ({row['culture_gap_score']}점 / {row['risk_level']})")
    print(f"  유형: {row['region_type']}")
    print(f"  원인: {row['main_reasons']}")
    print(f"  추천: {row['policy_recommendations']}")